# Introducción al cálculo proposicional con `calcprop`

Este cuaderno presenta las ideas básicas del cálculo proposicional (proposiciones, conectivos,
tablas de verdad, consecuencia lógica) y muestra, paso a paso, cómo las implementa y las usa la
librería `calcprop`. Cada concepto va acompañado del código que lo pone en práctica, y al final se
ve el *razonamiento* que sigue `calcprop` para decidir si una afirmación se deduce de unas
premisas.

Para ejecutarlo hace falta tener instalada la librería (`pip install calcprop`). El código fuente
comentado está en `CalcProp.org`, y el manual de uso en `Manual.org`.

In [1]:
import calcprop
assert hasattr(calcprop, 'explica'), (
    "Este cuaderno necesita calcprop >= 0.2.0, pero el kernel actual carga una versión "
    f"antigua desde {calcprop.__file__}. Actualiza con  pip install -U calcprop  "
    "o elige en Jupyter un kernel cuyo Python tenga la versión nueva.")
from calcprop import *

## 1. Proposiciones y conectivos

Una **proposición** es una afirmación de la que tiene sentido decir que es verdadera o falsa:
*«llueve»*, *«la matriz $A$ es invertible»*, *«el estimador es insesgado»*. El cálculo
proposicional no mira *dentro* de las proposiciones: las trata como átomos indivisibles y solo se
ocupa de cómo se combinan.

En `calcprop` una proposición atómica (o *variable proposicional*) se crea con `v()` y una
etiqueta cualquiera:

In [2]:
llueve = v('llueve')
paraguas = v('llevo paraguas')
llueve

v('llueve')

Las proposiciones se combinan mediante **conectivos**. Los cinco habituales, con su símbolo en
lógica y en `calcprop`:

| Conectivo     | Se lee                   | Lógica                  | `calcprop` |
|---------------|--------------------------|-------------------------|------------|
| negación      | no $P$                   | $\neg P$                | `-P`       |
| conjunción    | $P$ y $Q$                | $P \wedge Q$            | `P & Q`    |
| disyunción    | $P$ o $Q$ (o ambas)      | $P \vee Q$              | `P \| Q`   |
| implicación   | si $P$ entonces $Q$      | $P \rightarrow Q$       | `P >> Q`   |
| equivalencia  | $P$ si y solo si $Q$     | $P \leftrightarrow Q$   | `P ** Q`   |

Una **fórmula** es cualquier combinación de átomos y conectivos construida con estas reglas.

In [3]:
f = llueve >> paraguas          # «si llueve, llevo paraguas»
g = -(llueve & -paraguas)       # «no ocurre que llueva y no lleve paraguas»
print(f)
print(g)

v('llueve') >> v('llevo paraguas')
-(v('llueve') & -v('llevo paraguas'))


### La fórmula es un árbol

Internamente cada fórmula compuesta es una lista `[conectivo, operando, operando]`; los átomos
son las hojas. Esa estructura de árbol es lo único que necesita el resto de la librería.

In [4]:
print(g.data)
print(g.data[1].data)

['not', v('llueve') & -v('llevo paraguas')]
['and', v('llueve'), -v('llevo paraguas')]


### Dos advertencias prácticas

**Precedencia.** Los símbolos son los de Python, y Python los evalúa con *su* orden de
prioridad: `**` primero, luego `-`, luego `>>`, luego `&` y por último `|`. Ese orden no es el
habitual en lógica, así que `A & B >> C` se lee como `A & (B >> C)`. Regla práctica: **cuando se
mezclen conectivos distintos, poner paréntesis**. Si dudas, imprime la fórmula: `calcprop` la
muestra con los paréntesis que Python ha entendido.

In [5]:
A, B, C = v('A'), v('B'), v('C')
print(A & B >> C)        # ¡no es (A & B) >> C!
print((A & B) >> C)

v('A') & (v('B') >> v('C'))
(v('A') & v('B')) >> v('C')


**`and`, `or` y `not` no sirven.** Python no permite redefinir esas palabras, así que no
construyen fórmulas. Para que el error no pase desapercibido, `calcprop` lo detecta y avisa:

In [6]:
try:
    A and B
except TypeError as e:
    print("TypeError:", e)

TypeError: Una Proposicion no tiene valor de verdad por sí misma: use test(P, premisas). Para combinar proposiciones use los operadores &, | y - en lugar de and, or y not.


## 2. Mundos y valoraciones

Por sí sola, una fórmula no es verdadera ni falsa: depende de cómo sea el mundo. Fijar un
**mundo** es decidir el valor de verdad de cada átomo. Una vez fijado, el valor de cualquier
fórmula queda determinado por las tablas de los conectivos. La función que asigna así un valor a
cada fórmula se llama **valoración**; en `calcprop` es `span(F, P)`, donde el diccionario `F`
describe el mundo (los átomos que no aparecen en `F` se consideran falsos).

In [7]:
mundo = {llueve: True, paraguas: False}     # llueve y no llevo paraguas
print(span(mundo, llueve >> paraguas))     # la promesa «si llueve, llevo paraguas» se incumple
print(span(mundo, -(llueve & -paraguas)))  # la otra forma de decir lo mismo, también falsa

False
False


### Las tablas de los conectivos

Cada conectivo se define por su tabla: el valor de la fórmula compuesta en cada combinación de
valores de sus partes. La función `tabla_verdad` recorre todas las combinaciones (V primero) y
las muestra.

In [8]:
tabla_verdad(A & B)

v('A'),v('B'),v('A') & v('B')
V,V,V
V,F,F
F,V,F
F,F,F


In [9]:
tabla_verdad(A | B)

v('A'),v('B'),v('A') | v('B')
V,V,V
V,F,V
F,V,V
F,F,F


In [10]:
tabla_verdad(A >> B)

v('A'),v('B'),v('A') >> v('B')
V,V,V
V,F,F
F,V,V
F,F,V


La tabla de la implicación sorprende la primera vez: $A \rightarrow B$ **solo es falsa cuando
$A$ es verdadera y $B$ falsa**. Si $A$ es falsa, la implicación es verdadera sea cual sea $B$.
Piénsese en una promesa: «si llueve, llevo paraguas» solo se incumple si llueve y no lo llevo;
un día sin lluvia no la incumple. Por eso $A \rightarrow B$ equivale a $\neg A \vee B$:

In [11]:
tabla_verdad(-A | B)

v('A'),v('B'),-v('A') | v('B')
V,V,V
V,F,F
F,V,V
F,F,V


In [12]:
tabla_verdad(A ** B)

v('A'),v('B'),v('A') ** v('B')
V,V,V
V,F,F
F,V,F
F,F,V


## 3. Tablas de verdad de fórmulas compuestas

Con $n$ átomos hay $2^n$ mundos posibles. La tabla de verdad de una fórmula compuesta recorre
todos ellos, calculando de dentro hacia fuera; con `subformulas=True` se muestran las columnas
intermedias, como en las tablas que se construyen a mano.

In [13]:
tabla_verdad((A | B) >> C, subformulas=True)

v('A'),v('B'),v('C'),v('A') | v('B'),(v('A') | v('B')) >> v('C')
V,V,V,V,V
V,V,F,V,F
V,F,V,V,V
V,F,F,V,F
F,V,V,V,V
F,V,F,V,F
F,F,V,F,V
F,F,F,F,V


Según la última columna, una fórmula es:

- una **tautología** si es V en todos los mundos (es verdadera por su pura forma lógica),
- una **contradicción** si es F en todos,
- **contingente** si depende del mundo.

In [14]:
tabla_verdad(A | -A)          # principio del tercero excluido: tautología

v('A'),v('A') | -v('A')
V,V
F,V


In [15]:
tabla_verdad((A >> B) ** (-B >> -A), subformulas=True)   # contraposición: tautología

v('A'),v('B'),v('A') >> v('B'),-v('B'),-v('A'),-v('B') >> -v('A'),(v('A') >> v('B')) ** (-v('B') >> -v('A'))
V,V,V,F,F,V,V
V,F,F,V,F,F,V
F,V,V,F,V,V,V
F,F,V,V,V,V,V


In [16]:
tabla_verdad(A & -A)          # contradicción

v('A'),v('A') & -v('A')
V,F
F,F


**Ejercicio.** Comprueba con `tabla_verdad` cuáles de estas fórmulas son tautologías:
`(A >> B) >> (B >> A)`, `-(A & B) ** (-A | -B)` (ley de De Morgan), `(A >> B) | (B >> A)`.

In [17]:
t = tabla_verdad(-(A & B) ** (-A | -B))
print(t.tautologia)

True


## 4. Consecuencia lógica

En la práctica casi nunca preguntamos si una fórmula es verdadera, sino si **se deduce** de
otras. Decimos que $P$ es **consecuencia lógica** de las premisas $H_1, \dots, H_n$ (y escribimos
$H_1, \dots, H_n \models P$) cuando *en todo mundo en que las premisas son verdaderas, $P$ también
lo es*. Equivalentemente: **no existe ningún contraejemplo**, esto es, ningún mundo con las
premisas V y $P$ F.

Con la tabla de verdad basta mirar las filas donde todas las premisas valen V. `tabla_verdad`
las marca con `*`:

In [18]:
tabla_verdad(B, [A, A >> B])        # modus ponens: de A y «si A entonces B» se deduce B

,v('A'),v('B'),v('A') >> v('B'),v('B')
*,V,V,V,V
,V,F,F,F
,F,V,V,V
,F,F,V,F


En la única fila con premisas verdaderas, `B` es V: se deduce. Esto es exactamente lo que
responde `test(P, premisas)`, la función central de `calcprop`:

In [19]:
test(B, [A, A >> B])

True

Otros razonamientos clásicos, válidos e inválidos:

In [20]:
print("modus tollens        (-B, A>>B  ⊨ -A):", test(-A, [-B, A >> B]))
print("silogismo hipotético (A>>B, B>>C ⊨ A>>C):", test(A >> C, [A >> B, B >> C]))
print("afirmar consecuente  (B, A>>B  ⊨ A)?:", test(A, [B, A >> B]))
print("negar antecedente    (-A, A>>B ⊨ -B)?:", test(-B, [-A, A >> B]))

modus tollens        (-B, A>>B  ⊨ -A): True
silogismo hipotético (A>>B, B>>C ⊨ A>>C): True
afirmar consecuente  (B, A>>B  ⊨ A)?: False
negar antecedente    (-A, A>>B ⊨ -B)?: False


Las dos últimas son **falacias**: `test` devuelve `False` porque existe un contraejemplo. La
tabla lo señala en rojo (fila con premisas V y fórmula F):

In [21]:
tabla_verdad(A, [B, A >> B])

,v('A'),v('B'),v('A') >> v('B'),v('A')
*,V,V,V,V
,V,F,F,V
*,F,V,V,F
,F,F,V,F


Es importante entender qué significa `test(P, H) == False`: **no** que $P$ sea falsa, sino que
las premisas no bastan para asegurarla. Puede que se deduzca $\neg P$, o puede que las premisas
no digan nada sobre $P$. Volveremos sobre esto en la sección 6.

## 5. Cómo razona `calcprop`: tableaux semánticos

Recorrer los $2^n$ mundos funciona para fórmulas pequeñas, pero con 30 átomos son más de mil
millones de filas. `calcprop` usa en su lugar el método de los **tableaux semánticos** (o
*árboles de refutación*), que es lo que hace un lógico con lápiz y papel:

1. Para ver si $H \models P$, **supone que existe un contraejemplo**: las premisas son V y $P$ es F.
2. Descompone cada suposición sobre una fórmula compuesta en suposiciones sobre sus partes,
   aplicando **reglas que se leen de las tablas de verdad**. Algunas reglas dan una sola
   consecuencia («si $A \wedge B$ es V, entonces $A$ es V y $B$ es V»); otras **bifurcan** el
   razonamiento en ramas alternativas («si $A \wedge B$ es F, o bien $A$ es F, o bien $B$ es F»).
3. Cuando llega a un átomo, **anota** su valor. Si la misma rama exige después el valor
   contrario, la rama **se cierra**: era imposible.
4. Si **todas** las ramas se cierran, el contraejemplo no existe: $P$ se deduce. Si alguna rama
   queda abierta, sus anotaciones son un contraejemplo: $P$ no se deduce.

La función `explica` muestra ese árbol tal como `calcprop` lo construye:

In [22]:
explica(B, [A, A >> B])

¿Se deduce  v('B')  de las premisas  v('A'), v('A') >> v('B') ?
Método: buscamos un contraejemplo, es decir, un mundo en el que las premisas sean V y la fórmula F. Si toda rama se cierra por contradicción, no existe tal mundo y la fórmula se deduce.

Buscamos un mundo en el que v('B') = F, v('A') = V, v('A') >> v('B') = V:
v('B') = F  → anotamos v('B') = F.
v('A') = V  → anotamos v('A') = V.
v('A') >> v('B') = V  ⇒  por la regla de >>: o bien v('A') = F, o bien v('B') = V
    Rama 1 de 2: suponemos v('A') = F
        v('A') = F  ✗ contradice lo anotado (v('A') = V): rama cerrada.
    Rama 2 de 2: suponemos v('B') = V
        v('B') = V  ✗ contradice lo anotado (v('B') = F): rama cerrada.
    ✗ Las 2 ramas se cierran.

Conclusión: todas las ramas se cierran, no hay contraejemplo. v('B') se deduce de las premisas: test → True.

Cuando el razonamiento no cierra todas las ramas, `explica` entrega el contraejemplo:

In [23]:
explica(A >> C, [A >> B])

¿Se deduce  v('A') >> v('C')  de las premisas  v('A') >> v('B') ?
Método: buscamos un contraejemplo, es decir, un mundo en el que las premisas sean V y la fórmula F. Si toda rama se cierra por contradicción, no existe tal mundo y la fórmula se deduce.

Buscamos un mundo en el que v('A') >> v('C') = F, v('A') >> v('B') = V:
v('A') >> v('C') = F  ⇒  por la regla de >>: v('A') = V y v('C') = F
v('A') = V  → anotamos v('A') = V.
v('C') = F  → anotamos v('C') = F.
v('A') >> v('B') = V  ⇒  por la regla de >>: o bien v('A') = F, o bien v('B') = V
    Rama 1 de 2: suponemos v('A') = F
        v('A') = F  ✗ contradice lo anotado (v('A') = V): rama cerrada.
    Rama 2 de 2: suponemos v('B') = V
        v('B') = V  → anotamos v('B') = V.
        ✓ No quedan suposiciones por cumplir: el mundo { v('A') = V, v('C') = F, v('B') = V } las satisface todas.

Conclusión: el mundo { v('A') = V, v('C') = F, v('B') = V } es un contraejemplo (los átomos no listados pueden tomar cualquier valor). v('A') >> v('C') no se deduce de las premisas: test → False.

In [24]:
r = explica(A >> C, [A >> B])
print(r.resultado)        # el valor de test(...)
print(r.contraejemplo)    # el mundo que refuta la deducción

False
{v('A'): True, v('C'): False, v('B'): True}


### Las reglas son código

Cada regla del tableau es una pequeña función de la librería que recibe la fórmula y el valor
que debe tomar, y devuelve la lista de ramas (cada rama, una lista de suposiciones
`(subfórmula, valor)`). Compárese con la tabla de verdad de la conjunción de la sección 2:

In [25]:
import inspect, calcprop
print(inspect.getsource(calcprop._regla_and))
print(inspect.getsource(calcprop._regla_implica))

def _regla_and(P, VF):
    A, B = P.data[1], P.data[2]
    if VF:
        return [[(A, True), (B, True)]]
    return [[(A, False)], [(B, False)]]

def _regla_implica(P, VF):
    A, B = P.data[1], P.data[2]
    if VF:
        return [[(A, False)], [(B, True)]]
    return [[(A, True), (B, False)]]



La equivalencia bifurca en ambos casos, y por eso las fórmulas con muchas equivalencias anidadas
son las más costosas:

In [26]:
explica((A ** B) ** (B ** A))

¿Es  (v('A') ** v('B')) ** (v('B') ** v('A'))  una tautología (se deduce sin premisas)?
Método: buscamos un contraejemplo, es decir, un mundo en el que la fórmula sea F. Si toda rama se cierra por contradicción, no existe tal mundo y la fórmula es una tautología.

Buscamos un mundo en el que (v('A') ** v('B')) ** (v('B') ** v('A')) = F:
(v('A') ** v('B')) ** (v('B') ** v('A')) = F  ⇒  por la regla de **: o bien v('A') ** v('B') = V y v('B') ** v('A') = F, o bien v('A') ** v('B') = F y v('B') ** v('A') = V
    Rama 1 de 2: suponemos v('A') ** v('B') = V y v('B') ** v('A') = F
        v('A') ** v('B') = V  ⇒  por la regla de **: o bien v('A') = V y v('B') = V, o bien v('A') = F y v('B') = F
            Rama 1 de 2: suponemos v('A') = V y v('B') = V
                v('A') = V  → anotamos v('A') = V.
                v('B') = V  → anotamos v('B') = V.
                v('B') ** v('A') = F  ⇒  por la regla de **: o bien v('B') = V y v('A') = F, o bien v('B') = F y v('A') = V
                    Rama 1 de 2: suponemos v('B') = V y v('A') = F
                        v('B') = V  ✓ ya estaba anotado.
                        v('A') = F  ✗ contradice lo anotado (v('A') = V): rama cerrada.
                    Rama 2 de 2: suponemos v('B') = F y v('A') = V
                        v('B') = F  ✗ contradice lo anotado (v('B') = V): rama cerrada.
                    ✗ Las 2 ramas se cierran.
            Rama 2 de 2: suponemos v('A') = F y v('B') = F
                v('A') = F  → anotamos v('A') = F.
                v('B') = F  → anotamos v('B') = F.
                v('B') ** v('A') = F  ⇒  por la regla de **: o bien v('B') = V y v('A') = F, o bien v('B') = F y v('A') = V
                    Rama 1 de 2: suponemos v('B') = V y v('A') = F
                        v('B') = V  ✗ contradice lo anotado (v('B') = F): rama cerrada.
                    Rama 2 de 2: suponemos v('B') = F y v('A') = V
                        v('B') = F  ✓ ya estaba anotado.
                        v('A') = V  ✗ contradice lo anotado (v('A') = F): rama cerrada.
                    ✗ Las 2 ramas se cierran.
            ✗ Las 2 ramas se cierran.
    Rama 2 de 2: suponemos v('A') ** v('B') = F y v('B') ** v('A') = V
        v('A') ** v('B') = F  ⇒  por la regla de **: o bien v('A') = V y v('B') = F, o bien v('A') = F y v('B') = V
            Rama 1 de 2: suponemos v('A') = V y v('B') = F
                v('A') = V  → anotamos v('A') = V.
                v('B') = F  → anotamos v('B') = F.
                v('B') ** v('A') = V  ⇒  por la regla de **: o bien v('B') = V y v('A') = V, o bien v('B') = F y v('A') = F
                    Rama 1 de 2: suponemos v('B') = V y v('A') = V
                        v('B') = V  ✗ contradice lo anotado (v('B') = F): rama cerrada.
                    Rama 2 de 2: suponemos v('B') = F y v('A') = F
                        v('B') = F  ✓ ya estaba anotado.
                        v('A') = F  ✗ contradice lo anotado (v('A') = V): rama cerrada.
                    ✗ Las 2 ramas se cierran.
            Rama 2 de 2: suponemos v('A') = F y v('B') = V
                v('A') = F  → anotamos v('A') = F.
                v('B') = V  → anotamos v('B') = V.
                v('B') ** v('A') = V  ⇒  por la regla de **: o bien v('B') = V y v('A') = V, o bien v('B') = F y v('A') = F
                    Rama 1 de 2: suponemos v('B') = V y v('A') = V
                        v('B') = V  ✓ ya estaba anotado.
                        v('A') = V  ✗ contradice lo anotado (v('A') = F): rama cerrada.
                    Rama 2 de 2: suponemos v('B') = F y v('A') = F
                        v('B') = F  ✗ contradice lo anotado (v('B') = V): rama cerrada.
                    ✗ Las 2 ramas se cierran.
            ✗ Las 2 ramas se cierran.
    ✗ Las 2 ramas se cierran.

Conclusión: todas las ramas se cierran, no hay contraejemplo. (v('A') ** v('B')) ** (v('B') ** v('A')) se deduce de las premisas: test → True.

### Tableaux y tablas de verdad coinciden

Ambos métodos responden a la misma pregunta por caminos distintos, así que deben dar siempre el
mismo resultado. Es una buena comprobación de la librería (y así lo hacen sus pruebas
automáticas):

In [27]:
casos = [(B, [A, A >> B]), (A, [B, A >> B]), (A >> C, [A >> B, B >> C]),
         ((A | B) & -C, [A ** C]), (unoDe(A, B, C), [A, -B])]
for P, H in casos:
    print(test(P, H) == tabla_verdad(P, H).consecuencia, end=" ")

True True True True True 

Y donde la tabla es inabordable, el tableau sigue siendo inmediato:

In [28]:
import time
xs = [v(f'x{i}') for i in range(40)]              # 40 átomos: 2**40 ≈ 10**12 mundos
cadena = [xs[i] >> xs[i + 1] for i in range(39)]  # x0>>x1, x1>>x2, ..., x38>>x39
t0 = time.time()
print(test(xs[39], [xs[0]] + cadena), f"({time.time() - t0:.3f} s)")

True (0.001 s)


## 6. Verdadero, falso... o indecidible: `noDerivable`

Con unas premisas dadas, una afirmación $P$ puede estar en tres situaciones:

- se deduce $P$ (es *verdadera* dadas las premisas),
- se deduce $\neg P$ (es *falsa* dadas las premisas),
- no se deduce ni $P$ ni $\neg P$ (las premisas *no la deciden*).

`test` solo distingue la primera de las otras dos. Para preguntar directamente «¿es $P$
indecidible con lo que sé?» está `noDerivable(P)`, que es verdadera cuando $P$ **no** se deduce
de las premisas:

In [29]:
H = [A >> B, A]
print("¿se deduce B?              ", test(B, H))
print("¿se deduce C?              ", test(C, H))
print("¿se deduce -C?             ", test(-C, H))
print("¿es C indecidible?         ", test(noDerivable(C), H))
print("¿es B indecidible?         ", test(noDerivable(B), H))

¿se deduce B?               True
¿se deduce C?               False
¿se deduce -C?              False
¿es C indecidible?          True
¿es B indecidible?          False


`explica` muestra que la comprobación de `noDerivable` es un subproblema aparte (con su propio
tableau), cuyo resultado se incorpora al razonamiento principal:

In [30]:
explica(noDerivable(C), H)

¿Se deduce  noDerivable(v('C'))  de las premisas  v('A') >> v('B'), v('A') ?
Método: buscamos un contraejemplo, es decir, un mundo en el que las premisas sean V y la fórmula F. Si toda rama se cierra por contradicción, no existe tal mundo y la fórmula se deduce.

Buscamos un mundo en el que noDerivable(v('C')) = F, v('A') >> v('B') = V, v('A') = V:
noDerivable(v('C')) = F  ⇒  ¿se deduce v('C') de las premisas? Lo comprobamos aparte:
    Buscamos un mundo en el que v('C') = F, v('A') >> v('B') = V, v('A') = V:
    v('C') = F  → anotamos v('C') = F.
    v('A') >> v('B') = V  ⇒  por la regla de >>: o bien v('A') = F, o bien v('B') = V
        Rama 1 de 2: suponemos v('A') = F
            v('A') = F  → anotamos v('A') = F.
            v('A') = V  ✗ contradice lo anotado (v('A') = F): rama cerrada.
        Rama 2 de 2: suponemos v('B') = V
            v('B') = V  → anotamos v('B') = V.
            v('A') = V  → anotamos v('A') = V.
            ✓ No quedan suposiciones por cumplir: el mundo { v('C') = F, v('B') = V, v('A') = V } las satisface todas.
    Por tanto v('C') NO se deduce de las premisas, y noDerivable(v('C')) = V.
✗ noDerivable(v('C')) = F es imposible: rama cerrada.

Conclusión: todas las ramas se cierran, no hay contraejemplo. noDerivable(v('C')) se deduce de las premisas: test → True.

`noDerivable` no es un conectivo lógico sino una afirmación *sobre* las premisas; por eso puede
combinarse con otros conectivos en la fórmula (`-noDerivable(P)` significa «$P$ sí se deduce»),
pero **no** puede aparecer dentro de las premisas.

## 7. Dos conectivos de conveniencia: `alguno` y `unoDe`

`alguno(P, Q, R, ...)` es la disyunción de varias fórmulas («al menos una»), y `unoDe(P, Q, R, ...)`
expresa «exactamente una». Este último es muy útil para describir situaciones excluyentes: si
el enunciado sortea *uno* de varios supuestos, la negación de los demás pasa a ser deducible.

In [31]:
tabla_verdad(unoDe(A, B, C))

v('A'),v('B'),v('C'),"unoDe(v('A'), v('B'), v('C'))"
V,V,V,F
V,V,F,F
V,F,V,F
V,F,F,V
F,V,V,F
F,V,F,V
F,F,V,V
F,F,F,F


In [32]:
explica(-B, [unoDe(A, B, C), A])     # si exactamente uno es cierto y es A, entonces B es falso

¿Se deduce  -v('B')  de las premisas  unoDe(v('A'), v('B'), v('C')), v('A') ?
Método: buscamos un contraejemplo, es decir, un mundo en el que las premisas sean V y la fórmula F. Si toda rama se cierra por contradicción, no existe tal mundo y la fórmula se deduce.

Buscamos un mundo en el que -v('B') = F, unoDe(v('A'), v('B'), v('C')) = V, v('A') = V:
-v('B') = F  ⇒  por la regla de -: v('B') = V
v('B') = V  → anotamos v('B') = V.
unoDe(v('A'), v('B'), v('C')) = V  ⇒  por la regla de unoDe: o bien v('A') = V y v('B') = F y v('C') = F, o bien v('A') = F y v('B') = V y v('C') = F, o bien v('A') = F y v('B') = F y v('C') = V
    Rama 1 de 3: suponemos v('A') = V y v('B') = F y v('C') = F
        v('A') = V  → anotamos v('A') = V.
        v('B') = F  ✗ contradice lo anotado (v('B') = V): rama cerrada.
    Rama 2 de 3: suponemos v('A') = F y v('B') = V y v('C') = F
        v('A') = F  → anotamos v('A') = F.
        v('B') = V  ✓ ya estaba anotado.
        v('C') = F  → anotamos v('C') = F.
        v('A') = V  ✗ contradice lo anotado (v('A') = F): rama cerrada.
    Rama 3 de 3: suponemos v('A') = F y v('B') = F y v('C') = V
        v('A') = F  → anotamos v('A') = F.
        v('B') = F  ✗ contradice lo anotado (v('B') = V): rama cerrada.
    ✗ Las 3 ramas se cierran.

Conclusión: todas las ramas se cierran, no hay contraejemplo. -v('B') se deduce de las premisas: test → True.

## 8. Para qué sirve todo esto: bancos de preguntas

El paquete [`qbank`](https://github.com/mbujosab/calcprop-qbank) genera preguntas de opción
múltiple con variantes aleatorias. Cada pregunta se describe así:

- los **supuestos** del enunciado son fórmulas que se toman como premisas (por ejemplo,
  `unoDe(v('A es invertible'), v('A es singular'))` si el enunciado sortea una de las dos);
- cada **afirmación** ofrecida al estudiante lleva una fórmula (su *semántica*); la afirmación es
  correcta en una variante si `test(semántica, premisas)` es `True`;
- una **precondición** opcional decide si la afirmación tiene sentido en esa variante; con
  `noDerivable` se evita ofrecer una afirmación que el enunciado no permite decidir.

Así, el motor lógico garantiza que la clave de corrección de cada variante es consecuencia de lo
que dice su enunciado, y `explica` permite auditar por qué.

In [33]:
inv, sing, det0 = v('A es invertible'), v('A es singular'), v('det(A) = 0')
premisas = [unoDe(inv, sing), sing >> det0, det0 >> sing]
for afirmacion in [det0, -inv, inv | det0]:
    print(f"{afirmacion!r:45}", "correcta" if test(afirmacion, premisas + [sing]) else "incorrecta",
          "(si A es singular)")

v('det(A) = 0')                               correcta (si A es singular)
-v('A es invertible')                         correcta (si A es singular)
v('A es invertible') | v('det(A) = 0')        correcta (si A es singular)


## 9. Límites y lecturas

- `calcprop` es **proposicional**: no hay variables individuales ni cuantificadores («para todo»,
  «existe»). Todo lo que quiera decirse sobre objetos concretos debe reducirse a átomos.
- El método de los tableaux es **correcto y completo** para la lógica proposicional, pero de
  coste exponencial en el peor caso; para las fórmulas cortas de un banco de preguntas es
  irrelevante.
- El código completo, con su justificación, está en `CalcProp.org` (programación literaria: el
  fichero `.py` se genera de él). El manual de uso es `Manual.org`.